[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter15_overfitting_and_regularization/notebook.ipynb)

# Chapter 15: Overfitting and Regularization

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Chapter 11 left a row in a table unexplained. One of the five loss curve shapes was the one where training loss keeps falling while held out loss turns around and climbs, and the table said that Chapter 15 would deal with it. This is that chapter.

On 3 July 2012 five researchers at the University of Toronto posted a paper whose first sentence states the problem exactly: when a large network is trained on a small training set, it typically does badly on data it has not seen. Their fix was almost absurd. During training, throw away half the hidden units at random on every single example, so that no unit can rely on any other being there. They called it **dropout**.

Two of the five names on that paper, Alex Krizhevsky and Ilya Sutskever, were in the same months finishing the network from Chapter 1 that won ImageNet and started everything that followed. The two results came out of the same room.

Before you read on, try to guess the answers to these four questions.

1. **If a network has far more weights than it has training examples, what is to stop it from simply storing them?**
2. **How could randomly breaking your own network during training make it better?**
3. **Why would forcing the weights to stay small help at all?**
4. **If held out loss turns upward at some point, what is the obvious thing to do?**

## How a Network Memorizes

To study overfitting you have to cause it, and that means a dataset with genuine ambiguity in it. The plants from Chapter 1 come back, but this season the field is a mess: the two rings now overlap, so some plants simply cannot be identified from their position.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

X, y = make_circles(n_samples=500, noise=0.25, factor=0.5, random_state=7)
X_fit, X_test, y_fit, y_test = train_test_split(
    X, y, test_size=0.6, random_state=7, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_fit, y_fit, test_size=0.5, random_state=7, stratify=y_fit)
print(len(X_train), len(X_val), len(X_test))

Three splits, not two. Book 5 made the case and it holds here: the validation half is what you are allowed to look at while deciding things, and the test third is what you touch once, at the end. Every choice in this chapter is made against validation, and the test numbers appear only in the final table.

In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float32)
Xva = torch.tensor(X_val, dtype=torch.float32)
Xte = torch.tensor(X_test, dtype=torch.float32)
ytr, yva, yte = torch.tensor(y_train), torch.tensor(y_val), torch.tensor(y_test)
loss_fn = nn.CrossEntropyLoss()

def build(dropout=0.0, seed=0, width=256):
    torch.manual_seed(seed)
    layers = [nn.Linear(2, width), nn.ReLU()]
    if dropout:
        layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(width, width), nn.ReLU()]
    if dropout:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(width, 2))
    return nn.Sequential(*layers)

print(sum(p.numel() for p in build().parameters()), "parameters")

Sixty seven thousand weights, one hundred training examples. That is roughly 670 adjustable numbers per plant, which is more than enough to record every plant individually and ignore the pattern entirely. Chapter 3's parameter counting is doing real work here; a number you can compute in one line tells you in advance that this network can cheat.

One training function covers the whole chapter. Both regularizers are arguments that default to off, so every experiment below differs from the baseline by exactly one number, which is the discipline Chapter 13 used for optimizers and Chapter 12 used for batch sizes.

A word on time before you run it. This network is far larger than anything earlier in the book and 1200 epochs is a long run, so expect each call to take on the order of ten seconds on a laptop, and the chapter as a whole a couple of minutes. That is the price of a dataset small enough to overfit on purpose and a network big enough to do it.

In [ ]:
def run(dropout=0.0, weight_decay=0.0, epochs=1200, seed=0, lr=0.001):
    net = build(dropout, seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    train_hist, val_hist = [], []
    for epoch in range(epochs):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            batch_loss = loss_fn(net(xb), yb)
            batch_loss.backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            train_hist.append(loss_fn(net(Xtr), ytr).item())
            val_hist.append(loss_fn(net(Xva), yva).item())
    net.eval()
    with torch.no_grad():
        accuracy = lambda Xs, ys: (net(Xs).argmax(1) == ys).float().mean().item()
        return train_hist, val_hist, accuracy(Xtr, ytr), accuracy(Xte, yte)

Notice `net.train()` and `net.eval()` already in place. They do nothing at all for this first run and everything for the next section, so they go in now rather than being bolted on later.

In [ ]:
train_hist, val_hist, train_acc, test_acc = run()
print(round(train_hist[-1], 4), round(train_acc, 3))
print(round(val_hist[-1], 4), round(test_acc, 3))
print(round(min(val_hist), 4), val_hist.index(min(val_hist)))

Read the three lines in order. Training loss has fallen to 0.1009 and the network gets 97 percent of the training plants right. Validation loss ended at 1.1713. And the best validation loss it ever reached was 0.3459, at epoch 48, which means the model spent the remaining 1151 epochs getting steadily worse at the only thing that matters.

> 🖼️ **Figure 15.1:** The shape Chapter 11 promised. Training loss falls all the way down while validation loss bottoms out early and then climbs for the rest of the run.

This is what **overfitting** looks like when you plot it. The network is not failing. It is succeeding at the wrong thing: it has started fitting the noise in those hundred particular plants, and noise does not repeat in new plants, so every bit of effort spent on it makes the model worse everywhere else.

There are three honest ways out, and the rest of this chapter is one section each. Make the network less able to memorise, make large weights expensive, or simply stop at epoch 48.

## Dropout

Hinton's idea is best understood through what it prevents. In a large network, units settle into committees. One unit learns a feature that is only useful when a second unit fires at the same time, which is only useful when a third does, and the whole arrangement is tuned to the training set. The paper's word for this is **co-adaptation**.

Dropout breaks the committees by making membership unreliable. On every training example, each unit in a dropout layer is set to zero with probability $p$, independently. A unit cannot depend on a partner that might not be there, so it has to learn something useful on its own.

> 🖼️ **Figure 15.2:** Dropout during training. Each hidden unit is switched off at random for this one example, so the network that sees the next example is a slightly different network.

There is a detail here that trips up almost everyone, and it is why `net.train()` and `net.eval()` went in early. Dropout must happen while training and must not happen when you use the model. A prediction that changes every time you ask for it is useless. Watch both modes on the same input.

In [ ]:
torch.manual_seed(1)
drop = nn.Dropout(0.5)
x = torch.ones(8)
drop.train()
print([round(v, 1) for v in drop(x).tolist()])
drop.train()
print([round(v, 1) for v in drop(x).tolist()])
drop.eval()
print([round(v, 1) for v in drop(x).tolist()])

Two things to see. In training mode a different set of entries is zeroed each time, and the survivors come out as 2.0 rather than 1.0. PyTorch divides by $1-p$ so that the total going into the next layer stays about the same whether units were dropped or not, which is why you do not have to compensate yourself. In evaluation mode nothing is dropped and nothing is scaled, so every number passes through untouched and the model is deterministic again.

Forgetting `net.eval()` is one of the most common bugs in PyTorch code, and it does not announce itself with an error. It simply makes your evaluation noisy and your model look worse than it is.

In [ ]:
for p in [0.2, 0.5]:
    t, v, a_train, a_test = run(dropout=p)
    print(p, round(t[-1], 4), round(a_train, 3), round(v[-1], 4), round(a_test, 3))

At $p = 0.5$ the validation loss at the end of training falls from 1.1713 to 0.702 and test accuracy rises from 0.783 to 0.813. Look also at what happened to the training columns: training loss got *worse*, from 0.1009 to 0.2397, and training accuracy fell from 0.970 to 0.880. That is not a side effect to apologise for. It is the mechanism. You are deliberately giving up the ability to fit the training set perfectly, and buying generalisation with it.

## Weight Decay

The second route attacks the same problem from a different angle. To carve out the wiggly boundary that separates a hundred specific noisy points, a network needs large weights, because large weights are what make sharp turns possible. Make large weights expensive and the sharp turns go away.

**Weight decay** adds a penalty proportional to the size of every weight. In PyTorch it is one argument, already present in the `run` function above.

In [ ]:
for wd in [0.001, 0.01, 0.1]:
    t, v, a_train, a_test = run(weight_decay=wd)
    print(wd, round(t[-1], 4), round(a_train, 3), round(v[-1], 4), round(a_test, 3))

The middle setting is the best result in the chapter so far: validation loss 0.3709, which is close to the best the unregularised run ever managed at any point, except this run ends there rather than passing through. Test accuracy reaches 0.817.

The last row deserves a moment. At 0.1 the penalty overwhelms the task, every weight is squeezed towards zero, and both losses land on 0.6932 with accuracy exactly 0.500. That number is Chapter 5's know nothing baseline, $\ln 2 = 0.6931$. The network has been regularised into saying nothing at all, which is a useful reminder that every tool in this chapter can be turned up until it destroys the model.

Measuring the weights directly shows what the penalty does.

In [ ]:
for wd in [0.0, 0.01]:
    net = build(0.0, 0)
    opt = torch.optim.Adam(net.parameters(), lr=0.001, weight_decay=wd)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(0))
    for epoch in range(1200):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            batch_loss = loss_fn(net(xb), yb)
            batch_loss.backward()
            opt.step()
    size = sum(p.pow(2).sum().item() for p in net.parameters()) ** 0.5
    print(wd, round(size, 2))

Sixteen times smaller. Same architecture, same data, same optimizer, one extra argument. Figure 15.3 puts the three validation curves together.

> 🖼️ **Figure 15.3:** The same network under three regimes. Both regularizers flatten the climb; weight decay very nearly removes it on this problem.

## Early Stopping

The third route is the one Figure 15.1 was begging for all along. If validation loss bottoms out at epoch 48, stop at epoch 48.

The implementation has two parts that matter. You need **patience**, a number of epochs you are willing to wait through without improvement before giving up, because validation loss wobbles and the first uptick is not proof of anything. And you must keep a copy of the weights from the best epoch, because by the time you know the model has peaked, the live weights have already walked past it.

In [ ]:
import copy

def train_early(patience=50, max_epochs=1200, seed=0):
    net = build(0.0, seed)
    opt = torch.optim.Adam(net.parameters(), lr=0.001)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=16, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    best, best_state, best_epoch, waited = float("inf"), None, 0, 0
    for epoch in range(max_epochs):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            batch_loss = loss_fn(net(xb), yb)
            batch_loss.backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            val = loss_fn(net(Xva), yva).item()
        if val < best:
            best, best_epoch, waited = val, epoch, 0
            best_state = copy.deepcopy(net.state_dict())
        else:
            waited += 1
            if waited >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        accuracy = (net(Xte).argmax(1) == yte).float().mean().item()
    return epoch, best_epoch, best, accuracy

The `copy.deepcopy` matters. `state_dict()` hands back references to the live tensors, so without the copy your saved best would keep changing underneath you as training continued.

In [ ]:
stopped, best_epoch, best_val, accuracy = train_early()
print(stopped, best_epoch)
print(round(best_val, 4), round(accuracy, 3))

Training halted at epoch 98 having found its best at 48, and the restored model scores 0.830 on the test third. That is the highest number in the chapter, from the cheapest technique, in one twelfth of the training time. Table 15.1 puts everything side by side.

**Table 15.1: Every row trades training loss for validation loss, and the ordering of the two columns is almost exactly reversed. Fitting the training set better is not the goal and never was.**

| Setting | Train loss | Validation loss | Test accuracy |
|---|---|---|---|
| Nothing, 1200 epochs | 0.1009 | 1.1713 | 0.783 |
| Dropout 0.2 | 0.1574 | 0.9196 | 0.790 |
| Dropout 0.5 | 0.2397 | 0.702 | 0.813 |
| Weight decay 0.001 | 0.2347 | 0.4326 | 0.800 |
| Weight decay 0.01 | 0.3448 | 0.3709 | 0.817 |
| Early stopping, 98 epochs | — | 0.3459 | 0.830 |

Read the first two columns together one last time. From top to bottom, training loss rises steadily while validation loss falls just as steadily. If you had been choosing a model by training loss, you would have picked the worst row in the table.

Early stopping winning here is not a general law. It wins on this problem because the task is small and the overfitting is severe and early, so there is a clear peak to stop at. On larger problems the usual arrangement is all three at once: weight decay and dropout running throughout, with early stopping deciding when to quit. They do not compete.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Remove the cause.** Rebuild with `width=8` instead of 256 and train for the same 1200 epochs. Does a network with few enough weights overfit at all?
2. **Add data instead.** Raise `n_samples` to 2000 so the training half grows, keeping everything else fixed, and compare the gap with what dropout bought you.
3. **Break it on purpose.** Delete `net.eval()` from `run` and rerun the dropout rows. Explain the numbers you get.
4. **Combine them.** Try dropout 0.2 with weight decay 0.001 together and see whether the pair beats either alone.
5. **Test the patience.** Rerun `train_early` with patience 5 and with patience 200, and note which epoch each one stops at.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What does it mean to say a network has memorised the training set?**
2. **Why does dropout make training loss worse, and why is that acceptable?**
3. **What goes wrong if you forget to call `net.eval()` before predicting?**
4. **Why does penalising large weights reduce overfitting?**
5. **Why does early stopping need both patience and a saved copy of the weights?**
6. **Why must the stopping decision use validation data rather than test data?**

### Answers

1. That it has enough capacity to store the individual examples, including their noise, rather than the pattern behind them. Here 67,074 weights for 100 plants made that easy.
2. Because it removes units at random, so the network can no longer tune a perfect fit to the training set. That is the point: the fit it loses was a fit to noise.
3. Dropout stays active, so units are still being randomly zeroed. Predictions become random and the model appears worse than it is.
4. Because sharp, wiggly boundaries that separate individual noisy points require large weights. Keeping weights small keeps the boundary smooth.
5. Patience because validation loss wobbles and a single bad epoch is not a peak. The saved copy because the live weights keep moving past the best point while you wait to confirm it.
6. Because the test set would stop being an honest estimate the moment you used it to make a decision. It is spent once, at the very end.

## What Comes Next

You can now cause overfitting on purpose, recognise its shape on a loss curve, and reach for dropout, weight decay or early stopping with a reason for each.

- **Chapter 16, Making Training Stable,** covers the last two rows of Chapter 11's loss curve table, where the loss sits flat at the baseline or turns to `nan`, and the learning rate is not to blame.
- **Chapter 20, Your First Deep Learning Project,** returns to choosing a width, now that you have seen what too much capacity does.

Chapter 16 starts where Chapter 4 stopped, with gradients that fade to nothing on their way back through a deep network, and the two fixes that made very deep networks trainable at all.